# 05 - IEEE-CIS Rule Explanation Evaluation

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
KAGGLE = Path("/kaggle").exists()
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print({"project_root": str(PROJECT_ROOT), "quick_run": QUICK_RUN, "kaggle": KAGGLE})

In [ ]:
from src.experiment import run_predictive_benchmarks
from src.explanation import RuleExplainer, bootstrap_explanation_precision_gain, explanation_quality_metrics, rule_quality_table
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "04_explanation_evaluation"
result = run_predictive_benchmarks(
    PROJECT_ROOT / "configs/ieee_cis.yaml",
    output_dir=output_dir,
    model_names=("tree",),
    quick_run=QUICK_RUN,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
)
config = result["config"]
prepared = result["prepared"]
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
probabilities = result["test_probabilities"]["tree"]
threshold = result["thresholds"]["tree"]
explainer = RuleExplainer(
    engine,
    config["logic"]["activation_threshold"],
    config["logic"]["top_k_rules"],
)
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
explanations["y_true"] = prepared.y_test
print({"data_source": result["data_source"], "threshold": threshold})
display(explanations.head())

## Data

In [ ]:
truth = engine.evaluate(prepared.test_frame)
rule_quality = rule_quality_table(truth, prepared.y_test, config["logic"]["activation_threshold"])
display(rule_quality.round(4))

## Results

In [ ]:
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations,
    prepared.y_test,
    probabilities,
    threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"],
    seed=config["project"]["seed"],
))
quality_frame = pd.DataFrame([quality])
display(quality_frame.round(4))

predicted_alert = probabilities >= threshold
case_frame = explanations.copy()
case_frame["case_type"] = np.select(
    [predicted_alert & (prepared.y_test == 1), predicted_alert & (prepared.y_test == 0), (~predicted_alert) & (prepared.y_test == 1)],
    ["true_positive", "false_positive", "false_negative"],
    default="true_negative",
)
selected_cases = pd.concat([
    group.sort_values("predicted_probability", ascending=False).head(3)
    for _, group in case_frame.groupby("case_type")
]).sort_values(["case_type", "predicted_probability"], ascending=[True, False])
display(selected_cases[["case_type", "y_true", "predicted_probability", "rule_names", "rule_strengths", "explanation"]])

quality_frame.to_csv(output_dir / "explanation_quality.csv", index=False)
rule_quality.to_csv(output_dir / "test_rule_quality.csv", index=False)
selected_cases.to_json(output_dir / "explanation_cases.json", orient="records", indent=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(explanations["max_rule_strength"], bins=30, ax=axes[0], color="#4C72B0")
axes[0].axvline(config["logic"]["activation_threshold"], color="black", linestyle="--")
axes[0].set_title("Maximum rule strength")
sns.countplot(data=case_frame, x="case_type", hue="explained", ax=axes[1])
axes[1].set_title("Explained status by outcome")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(output_dir / "explanation_overview.png", dpi=160, bbox_inches="tight")
plt.show()

## Takeaways

In [ ]:
display(Markdown(
    f"- Test alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**.\n"
    f"- Explained-alert precision gain: **{quality['explained_alert_precision_gain']:.3f}** (95% bootstrap CI **[{quality['precision_gain_ci_low']:.3f}, {quality['precision_gain_ci_high']:.3f}]**).\n"
    f"- Mean rule count: **{quality['mean_rule_count']:.2f}**.\n"
    f"- Prediction-rule consistency: **{quality['prediction_rule_consistency']:.3f}**.\n"
    f"- Contradiction rate: **{quality['contradiction_rate']:.3f}**.\n"
    "- These values quantify rule evidence behavior; they do not establish causal explanations."
))